# Surface water from SAR — CSA adaptation
Role: `demostracion_registro_corto`.
Adapted raster reading, masked arrays and CRS inspection from [CSA tutorial](https://github.com/asc-csa/Synthetic_Aperture_Radar_Tutorial), Option 2 Python, BC Wildfire raster plot. MIT; copyright Canadian Space Agency 2022; see LICENSE-CSA.txt. Water thresholding is an extension, not a CSA-validated classifier.

Install numpy, rasterio, matplotlib. Obtain RCM through EODMS or Sentinel-1 through ASF/Earthdata (catalog in data/recursos/csa). Never process the tutorial preview image as calibrated backscatter. In SNAP apply orbit, thermal/border-noise removal, radiometric calibration, speckle filtering, terrain correction and convert sigma0 to dB. Fix polarization, relative orbit, incidence range, pixel spacing, AOI and valid coverage across dates. Reproject to EPSG:6933, clip to a fixed lake/wetland polygon with nodata outside. Wind roughening, flooded vegetation, shadow and wet soil can bias the mask; validate against independent outlines. -17 dB is illustrative and must be calibrated locally.

Create a manifest JSON list with date (ISO day), path (calibrated clipped GeoTIFF), source, product_id and polarization. Set SAR_MANIFEST to its path. Without actual imagery the notebook reports blocked and does not fabricate areas.

In [ ]:
import os, sys, json, csv
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'backend/clima_nic').exists())
sys.path.insert(0, str(root/'backend'))
from clima_nic.fuentes.sar_agua import procesar
manifest = os.environ.get('SAR_MANIFEST')
rows = []
if not manifest:
    print('BLOCKED: set SAR_MANIFEST to real calibrated imagery manifest; no measured area produced.')
else:
    items = json.loads(Path(manifest).read_text())
    output = Path(os.environ.get('SAR_OUTPUT', 'sar-output'))
    output.mkdir(parents=True, exist_ok=True)
    for item in items:
        for key in ['date','path','source','product_id','polarization']:
            if not item.get(key): raise ValueError('Missing '+key)
        mask = output / (item['product_id']+'_water.tif')
        rows.append({**item, **procesar(item['path'], mask), 'mask':str(mask)})
    if rows:
        with (output/'water_area.csv').open('w', newline='') as f:
            writer=csv.DictWriter(f, fieldnames=rows[0].keys()); writer.writeheader(); writer.writerows(rows)
    print(rows)


## Real ASF OPERA demonstration
`opera_demostracion.py` downloads authenticated OPERA_L2_RTC-S1_V1 VV gamma0 power for 2024-03-04, 2024-04-21 and 2024-07-14 on track T092. It mosaics bursts without double counting, averages power to 100 m EPSG:6933, then converts with 10 log10. Unlike the sigma0-dB input above, these are gamma0 products: no conversion to sigma0 is claimed. Common valid pixels define the fixed observed domain. Full-lake extent is not inferred from partial coverage. Threshold sensitivity is evaluated at -19, -17 and -15 dB; this is not a validated shoreline classifier.
Source/method: https://hyp3-docs.asf.alaska.edu/guides/opera_rtc_product_guide/ ; lake boundary: Natural Earth 10m lakes, public domain. Rectangles for Los Guatuzos and San Juan are analytical windows, not protected-area boundaries.
Set `SAR_RUN_OPERA=1` to reproduce acquisition and outputs after Earthdata/ASF authorization. Originals and masks remain outside Git; derived areas, provenance and same-calendar-day SWOT comparisons are saved in data/recursos/csa. Same-day does not mean simultaneous acquisitions.


In [ ]:
import subprocess
if os.environ.get("SAR_RUN_OPERA") == "1":
    subprocess.run([sys.executable, str(root/"scripts/sar/opera_demostracion.py")], cwd=root, check=True)
else:
    print("Set SAR_RUN_OPERA=1 to run authenticated real-data OPERA processing.")


## Independent optical validation and incomplete status
Run `python scripts/sar/calibrar_optica.py` to acquire PO.DAAC OPERA DSWx-HLS BWTR reference (+/-2 days), calibrate on training stripes and evaluate on separated held-out stripes. The provisional optical product is not field truth. Raw optical rasters remain outside Git.
See sar_validacion_optica.json and sar_superficie_calibrada.csv: March improves, April/July fail. Common clear reference inside the lake is only 0.02 km² across three dates. All surface publication gates fail, so validated water-area fields remain blank. Do not use candidate areas for a water trend or a physical SWOT height–area relationship. Original unsuccessful experiments are retained, not suppressed.
